# Question Router: TF-IDF + Logistic Regression

Train a lightweight question-only router for `chartqa`, `docvqa`, and `textvqa`, then map each predicted task to the current best backend:

- `chartqa -> chart_dora_r8_a16_B_lr2e-5`
- `docvqa -> base_zero_shot`
- `textvqa -> textvqa_lora_only`

In [ ]:
from pathlib import Path
import json
import random
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("/content/multi-task-moe-vlm-assistant")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data/processed/multitask/validation.jsonl"
ROUTER_DIR = PROJECT_ROOT / "checkpoints/router"
ROUTER_PATH = ROUTER_DIR / "question_router.joblib"
METADATA_PATH = ROUTER_DIR / "question_router_metadata.json"
SEED = 42
TEST_SIZE = 0.2
MIN_CONFIDENCE = 0.65

random.seed(SEED)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH exists:", DATA_PATH.exists())

## Load Questions

In [ ]:
from collections import Counter
from src.data.answers import canonicalize_task_type


def load_router_records(path: Path) -> list[dict]:
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            question = str(row.get("question", "")).strip()
            if not question:
                continue
            task_type = canonicalize_task_type(row.get("task_type", ""))
            if task_type not in {"chartqa", "docvqa", "textvqa"}:
                continue
            records.append({"question": question, "task_type": task_type})
    return records

records = load_router_records(DATA_PATH)
print("records:", len(records))
print("task counts:", Counter(r["task_type"] for r in records))
records[:3]

## Split

In [ ]:
from sklearn.model_selection import train_test_split

questions = [r["question"] for r in records]
labels = [r["task_type"] for r in records]

train_q, test_q, train_y, test_y = train_test_split(
    questions,
    labels,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=labels,
)

print("train:", len(train_q), Counter(train_y))
print("test:", len(test_q), Counter(test_y))

## Train TF-IDF + Logistic Regression

In [ ]:
from src.routing.task_router import (
    TfidfLogRegTaskRouter,
    build_tfidf_logreg_pipeline,
    select_task_backend,
    summarize_router_backends,
)

router = TfidfLogRegTaskRouter()
router.fit(train_q, train_y)
print(router.pipeline)

## Evaluate

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

pred_y = [router.predict(q) for q in test_q]
accuracy = accuracy_score(test_y, pred_y)
macro_f1 = f1_score(test_y, pred_y, average="macro")

print(f"accuracy: {accuracy:.4f}")
print(f"macro_f1: {macro_f1:.4f}")
print(classification_report(test_y, pred_y, digits=4))

labels_order = ["chartqa", "docvqa", "textvqa"]
cm = confusion_matrix(test_y, pred_y, labels=labels_order)
pd.DataFrame(cm, index=[f"true_{x}" for x in labels_order], columns=[f"pred_{x}" for x in labels_order])

## Backend Decisions

In [ ]:
decisions = [select_task_backend(q, router=router, min_confidence=MIN_CONFIDENCE) for q in test_q]
print("backend counts:", summarize_router_backends(decisions))

preview = []
for q, true_label, pred_label, decision in list(zip(test_q, test_y, pred_y, decisions))[:20]:
    preview.append(
        {
            "question": q,
            "true_task": true_label,
            "pred_task": pred_label,
            "backend": decision.backend_name,
            "use_adapter": decision.use_adapter,
            "adapter": decision.adapter_name,
            "confidence": decision.confidence,
        }
    )

pd.DataFrame(preview)

## Low Confidence Fallback Check

In [ ]:
for q in [
    "what is shown here?",
    "what is the invoice date?",
    "what is the highest value in 2020?",
    "what word is written on the sign?",
]:
    decision = select_task_backend(q, router=router, min_confidence=MIN_CONFIDENCE)
    print(q)
    print(" ->", decision)

## Save Router

In [ ]:
ROUTER_DIR.mkdir(parents=True, exist_ok=True)
router.save(ROUTER_PATH)

metadata = {
    "model_type": "tfidf_logistic_regression",
    "router_path": str(ROUTER_PATH),
    "data_path": str(DATA_PATH),
    "seed": SEED,
    "test_size": TEST_SIZE,
    "min_confidence": MIN_CONFIDENCE,
    "accuracy": accuracy,
    "macro_f1": macro_f1,
    "backend_policy": {
        "chartqa": "chart_dora_r8_a16_B_lr2e-5",
        "docvqa": "base_zero_shot",
        "textvqa": "textvqa_lora_only",
    },
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("saved router:", ROUTER_PATH)
print("saved metadata:", METADATA_PATH)

## Load And Predict Helper

In [ ]:
loaded_router = TfidfLogRegTaskRouter.load(ROUTER_PATH)


def predict_task(question: str, min_confidence: float = MIN_CONFIDENCE) -> dict:
    task_type, confidence = loaded_router.predict_with_confidence(question)
    decision = select_task_backend(question, router=loaded_router, min_confidence=min_confidence)
    return {
        "task_type": task_type,
        "confidence": confidence,
        "backend_name": decision.backend_name,
        "use_adapter": decision.use_adapter,
        "adapter_name": decision.adapter_name,
        "checkpoint_dir": decision.checkpoint_dir,
    }

predict_task("What is the total revenue in 2020?")